# GCN — Semi-Supervised Classification with Graph Convolutional Networks (2017)

_Papers · Graph Neural Networks_

**Each layer mixes every node with its neighbours through one fixed normalized-adjacency matrix, so a handful of labels spread across the whole graph.**

---

This notebook is a practice scaffold for the **GCN — Semi-Supervised Classification with Graph Convolutional Networks (2017)** lesson. We build it up one step at a time — the normalized adjacency, the layer, the model, the two-label training run — so each piece earns its place before the next. Run each cell, read the note above it, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## Reference implementation — PyTorch

We build a Graph Convolutional Network from raw tensors — no graph library. We go in four steps: (1) replay the lesson's hand-worked 3-node example, (2) build a tiny two-community graph and its normalized adjacency, (3) define the GCN layer and a 2-layer model, and (4) train semi-supervised from just two labels.

### Step 1 — Replay the worked 3-node example

Before any learning, we reproduce the lesson's by-hand calculation on a 3-node path graph `0-1-2`. The heart of a GCN layer is the *normalized adjacency* `S`: we add self-loops (`A-hat = A + I`), compute each node's degree, and symmetrically normalize by `D-hat^{-1/2} A-hat D-hat^{-1/2}`. Applying `sigma(S H W)` once mixes each node with its neighbours — here every value stays non-negative, so ReLU acts as the identity.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)

# One GCN layer on a 3-node path graph 0-1-2 (edges 0-1, 1-2).
A = torch.tensor([[0., 1., 0.],
                  [1., 0., 1.],
                  [0., 1., 0.]])

Ahat = A + torch.eye(3)              # A-hat = A + I_N: add self-loops   (Section 2.2)
deg = Ahat.sum(1)                    # d-hat_ii = sum_j A-hat_ij  ->  [2, 3, 2]
Dinv = torch.diag(deg.pow(-0.5))     # D-hat^{-1/2}
S = Dinv @ Ahat @ Dinv               # S = D-hat^{-1/2} A-hat D-hat^{-1/2}

H = torch.tensor([[1., 0.],          # node features (3 nodes x 2)
                  [0., 1.],
                  [1., 1.]])
W = torch.tensor([[1., -1.],         # layer weights (2 x 2)
                  [0., 2.]])

pre_activation = S @ H @ W           # Eq. 2 before the nonlinearity: S H W
out = torch.relu(pre_activation)     # Eq. 2:  sigma( S H W )

print("normalized adjacency S =\n", S.round(decimals=4))
print("S @ H @ W (pre-ReLU) =\n", pre_activation.round(decimals=4))
print("worked-example layer output =\n", out.round(decimals=4))
# S =  [[0.5000, 0.4082, 0.0000],
#       [0.4082, 0.3333, 0.4082],
#       [0.0000, 0.4082, 0.5000]]
# S@H@W = [[0.5000, 0.3165], [0.8165, 0.6667], [0.5000, 1.3165]]   (all >= 0, so ReLU is identity)

### Step 2 — Build a two-community graph and its normalized adjacency

Now a slightly larger synthetic graph: two clusters that are densely connected within and sparsely connected between (`p_in` >> `p_out`). The nodes are *featureless*, so we hand each one a one-hot identity row. We then build the renormalized adjacency `S` once — the same `D-hat^{-1/2} A-hat D-hat^{-1/2}` recipe from Step 1 — which is what propagates signal between neighbours.

In [ ]:
def make_two_community_graph(n_per=10, p_in=0.45, p_out=0.03, seed=1):
    g = torch.Generator().manual_seed(seed)
    n = 2 * n_per
    comm = torch.cat([torch.zeros(n_per), torch.ones(n_per)]).long()   # ground-truth community
    A = torch.zeros(n, n)
    for i in range(n):
        for j in range(i + 1, n):
            p = p_in if comm[i] == comm[j] else p_out
            if torch.rand(1, generator=g).item() < p:
                A[i, j] = A[j, i] = 1.0
    return A, comm

A, labels = make_two_community_graph()
n = A.shape[0]
X = torch.eye(n)                                 # featureless: one-hot identity features

# Build the renormalized adjacency S once (the heart of the paper).
def normalized_adjacency(A):
    Ah = A + torch.eye(A.shape[0])               # add self-loops:  A-hat = A + I_N
    d = Ah.sum(1)                                # degrees of A-hat
    Di = torch.diag(d.pow(-0.5))                 # D-hat^{-1/2}
    return Di @ Ah @ Di                          # D-hat^{-1/2} A-hat D-hat^{-1/2}

S = normalized_adjacency(A)
print("graph nodes:", n, " edges:", int(A.sum().item() // 2))

### Step 3 — Define the GCN layer and the 2-layer model

A GCN layer is just `S @ (H @ W)`: multiply features by a learnable weight, then mix across neighbours with `S` (Eq. 2, pre-activation). The full model stacks two such layers — a ReLU after the first, raw logits out of the second (softmax is folded into `cross_entropy`). Two layers means each node hears from neighbours up to two hops away.

In [ ]:
class GCNLayer(nn.Module):
    def __init__(self, c_in, c_out):
        super().__init__()
        self.W = nn.Parameter(torch.empty(c_in, c_out))
        nn.init.xavier_uniform_(self.W)

    def forward(self, S, H):
        transformed = H @ self.W                 # apply the learnable weight
        return S @ transformed                   # Eq. 2 (pre-activation):  S H W

class GCN(nn.Module):
    def __init__(self, c_in, c_hid, c_out):
        super().__init__()
        self.l1 = GCNLayer(c_in, c_hid)
        self.l2 = GCNLayer(c_hid, c_out)

    def forward(self, S, X):
        h = torch.relu(self.l1(S, X))            # Eq. 9:  ReLU( S X W0 )
        return self.l2(S, h)                     # logits; softmax handled by cross_entropy

### Step 4 — Train semi-supervised from two labels, then classify everyone

This is the paper's central trick: we reveal exactly **one** label per community and compute the loss over those two nodes only (Eq. 10). Because every layer averages each node with its neighbours, those two seed labels diffuse across the graph during training. After ~120 steps we classify *all* nodes — even though we only ever supervised two.

In [ ]:
# Reveal ONE label per community; train on those only.
train_idx = torch.tensor([0, n - 1])             # one seed per cluster

torch.manual_seed(0)
model = GCN(c_in=n, c_hid=8, c_out=2)
opt = torch.optim.Adam(model.parameters(), lr=0.05, weight_decay=5e-4)

for epoch in range(120):
    model.train()
    opt.zero_grad()
    logits = model(S, X)
    loss = F.cross_entropy(logits[train_idx], labels[train_idx])   # Eq. 10: loss over labeled set only
    loss.backward()
    opt.step()

# Classify ALL nodes (we only ever showed 2 labels).
model.eval()
with torch.no_grad():
    pred = model(S, X).argmax(1)

acc = (pred == labels).float().mean().item()
print(f"trained on {len(train_idx)} labels; full-graph node-classification accuracy = {acc:.3f}")
# trained on 2 labels; full-graph node-classification accuracy = 0.950
# (our small run on a synthetic graph -- not the paper's reported Cora/Citeseer/Pubmed numbers)

## Visualize the data & results

_From just 2 labels on a 30-node two-community graph, does the normalized-adjacency mixing (S) spread the labels &mdash; and what happens if we remove it (S &rarr; identity)?_

### Step 1 — Build a graph and two propagation matrices to compare

To test whether the neighbour mixing is what spreads labels, we build a fresh two-community graph and prepare *two* propagation matrices: the real normalized adjacency `S`, and the identity `I`. Running the same model with `I` is an honest ablation — it removes neighbour mixing while changing nothing else, turning each layer into a plain per-node MLP.

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F, numpy as np

# Does the normalized-adjacency mixing (S) actually spread a few labels? Compare a GCN
# that uses S against an ablation that replaces S with the identity (no neighbour mixing).
def make_graph(n_per=15, p_in=0.40, p_out=0.02, seed=2):
    g = torch.Generator().manual_seed(seed)
    n = 2 * n_per
    comm = torch.cat([torch.zeros(n_per), torch.ones(n_per)]).long()
    A = torch.zeros(n, n)
    for i in range(n):
        for j in range(i + 1, n):
            p = p_in if comm[i] == comm[j] else p_out
            if torch.rand(1, generator=g).item() < p:
                A[i, j] = A[j, i] = 1.0
    return A, comm

def norm_adj(A):
    Ah = A + torch.eye(A.shape[0])
    d = Ah.sum(1)
    Di = torch.diag(d.pow(-0.5))
    return Di @ Ah @ Di

A, labels = make_graph()
n = A.shape[0]
X = torch.eye(n)
S = norm_adj(A)                                  # real propagation matrix
I = torch.eye(n)                                 # ablation: no neighbour mixing
train_idx = torch.tensor([0, n - 1])             # one labeled node per community

### Step 2 — Define the model and a training run that records accuracy

The model is the same 2-layer GCN, but written so the propagation matrix `M` is passed in — that lets us swap `S` for `I` without touching anything else. The `run` helper trains for 60 epochs from the same seed and records full-graph accuracy after every epoch, so we can watch the learning curve.

In [ ]:
class GCN(nn.Module):
    def __init__(s):
        super().__init__()
        s.W0 = nn.Parameter(torch.empty(n, 8))
        s.W1 = nn.Parameter(torch.empty(8, 2))
        nn.init.xavier_uniform_(s.W0)
        nn.init.xavier_uniform_(s.W1)

    def forward(s, M, X):
        h = torch.relu(M @ (X @ s.W0))           # layer 1: mix with M, then ReLU
        return M @ (h @ s.W1)                     # layer 2: logits

def run(M, seed=0):
    torch.manual_seed(seed)
    m = GCN()
    opt = torch.optim.Adam(m.parameters(), lr=0.05, weight_decay=5e-4)
    accs = []
    for ep in range(60):
        m.train()
        opt.zero_grad()
        loss = F.cross_entropy(m(M, X)[train_idx], labels[train_idx])
        loss.backward()
        opt.step()
        with torch.no_grad():
            accs.append((m(M, X).argmax(1) == labels).float().mean().item())
    return accs

### Step 3 — Run both and compare the curves

We train once with `S` (the real GCN) and once with `I` (the ablation), then print accuracy at 30 sampled epochs. The GCN climbs to ~100% within a couple of epochs because two hops of averaging carry each seed label across its community; the ablation stays near chance because no node ever sees a neighbour.

In [ ]:
gcn = run(S)        # uses the graph (S)
mlp = run(I)        # ABLATION: identity instead of S -> no neighbour mixing

idx = np.linspace(0, 59, 30).astype(int)
print("GCN (uses S):", [[int(i), round(gcn[i], 3)] for i in idx])
print("Ablation (S->I):", [[int(i), round(mlp[i], 3)] for i in idx])
# GCN -> 100% within ~2 epochs; ablation stays near chance (~40-50%).

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** The ablation. You have a 2-layer GCN that labels a 20-node, two-community graph correctly
            from just 2 labels (one per community). Replace the propagation matrix $S$ with the identity
            $I_N$ (so each layer is relu(I @ H @ W) = relu(H @ W) &mdash; a plain per-node MLP with
            no neighbour mixing) and retrain. What happens to the 18 unlabeled nodes, and what does that
            demonstrate?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Swap one matrix: change S to torch.eye(n); keep the same weights, optimizer, labels, and epochs. — _An honest ablation changes exactly one thing &mdash; the neighbour mixing &mdash; so any difference is attributable to the graph convolution._
- Retrain and look at accuracy on the 18 unlabeled nodes: it sits near chance (~50%) and never climbs. — _With $S=I$, no node ever sees a neighbour, so the 18 unlabeled nodes receive zero label signal &mdash; the model can only fit the 2 it was shown._
- Conclude that the normalized adjacency $S$ (the message passing), not the weights, is what spreads the labels. — _Both models have identical learnable weights and capacity; only the one that mixes neighbours generalizes, isolating $S$ as the cause._

**Answer:** With $S=I_N$ the unlabeled nodes collapse to roughly chance accuracy &mdash; the model can
                 only memorize the 2 labeled nodes because nothing carries their signal outward. Restoring
                 $S=\hat{D}^{-1/2}\hat{A}\hat{D}^{-1/2}$ lets each layer average neighbours, so after two hops
                 every node has heard from a labeled one and the accuracy jumps to ~100%. Since the only change
                 was $S$ vs $I$, this isolates the graph convolution (message passing), not extra
                 parameters, as the reason a handful of labels generalizes. The CODEVIZ panel shows exactly this
                 contrast.

</details>

**Problem 2.** For the 3-node path graph in the worked example ($\hat{A}=\begin{bmatrix}1&1&0\\1&1&1\\0&1&1\end{bmatrix}$),
            compute the renormalized weight $S_{12}$ on the edge between the hub (node 1, degree 3) and node 2
            (degree 2). Why is it smaller than node 2's self-weight $S_{22}$?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Use $S_{ij}=\hat{A}_{ij}/\sqrt{\hat{d}_i\hat{d}_j}$. Here $\hat{A}_{12}=1$, $\hat{d}_1=3$, $\hat{d}_2=2$, so $S_{12}=1/\sqrt{3\cdot2}=1/\sqrt6\approx0.408$. — _The symmetric normalization divides each edge by the geometric mean of the two endpoints' degrees._
- Compute the self-weight: $S_{22}=\hat{A}_{22}/\sqrt{\hat{d}_2\hat{d}_2}=1/\sqrt{2\cdot2}=1/2=0.5$. — _A self-loop is an edge from a node to itself, normalized by its own degree on both sides._
- Compare: $0.408 \lt 0.5$, so the edge to the higher-degree hub is down-weighted relative to node 2's own contribution. — _Symmetric normalization deliberately damps connections to high-degree hubs, preventing them from dominating every neighbour's average._

**Answer:** $S_{12}=1/\sqrt{3\cdot2}=1/\sqrt6\approx0.408$, while node 2's self-weight is
                 $S_{22}=1/\sqrt{2\cdot2}=0.5$. The edge to the hub is smaller because the symmetric
                 normalization divides by $\sqrt{\hat{d}_i\hat{d}_j}$, so a connection to a high-degree node
                 (degree 3) is penalized more than a self-loop on a low-degree node (degree 2). This is exactly
                 the "don't let hubs dominate" behavior that makes GCN stable.

</details>

**Problem 3.** A friend builds a GCN and trains it for 12 layers "to capture long-range structure," but accuracy
            on a small graph gets worse than their 2-layer version. They have no bug. What is happening,
            and what is the fix?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Recall each layer multiplies by $S$ once, averaging every node with its neighbours one more hop. — _Repeated averaging is a smoothing (diffusion) operation on the graph._
- After many layers every node's representation has been averaged over most of the graph, so all node vectors converge toward the same value &mdash; they become indistinguishable. — _This is over-smoothing: the very mixing that helps at 2-3 hops erases per-node information when overdone._
- Reduce to 2-3 layers (or add residual connections / jumping-knowledge links if deep mixing is truly needed). — _A shallow stack mixes enough to spread labels without washing out each node's identity._

**Answer:** It is over-smoothing: because every GCN layer averages each node with its neighbours,
                 stacking 12 of them smooths the whole graph until all node representations look the same and
                 become inseparable, so accuracy collapses. The fix is to use only 2-3 layers (the typical GCN
                 depth), or, if many hops are genuinely needed, add residual / skip connections to preserve each
                 node's own signal.

</details>